# 🤖 Notebook 03 — Modelo Principal: K-Nearest Neighbors (KNN)

## Fundamentos Teóricos

### ¿Qué es KNN?
K-Nearest Neighbors (KNN) es un algoritmo de aprendizaje supervisado **no paramétrico** y **lazy** (perezoso):  
no construye un modelo explícito durante el entrenamiento, sino que memoriza el dataset y clasifica nuevas instancias en tiempo de predicción.

### Base Matemática — Distancia Euclidiana
Para dos puntos $\mathbf{x} = (x_1, x_2, \ldots, x_n)$ y $\mathbf{q} = (q_1, q_2, \ldots, q_n)$:

$$d(\mathbf{x}, \mathbf{q}) = \sqrt{\sum_{i=1}^{n}(x_i - q_i)^2}$$

### Proceso de Clasificación
1. Dado un punto de consulta $\mathbf{q}$, calcular $d(\mathbf{q}, \mathbf{x}_i)$ para todos los puntos de entrenamiento.
2. Seleccionar los **K vecinos más cercanos**.
3. Asignar la clase por **voto mayoritario**:

$$\hat{y} = \arg\max_{c} \sum_{i=1}^{K} \mathbb{1}[y_i = c]$$

### Hiperparámetro K — Bias-Variance Tradeoff
| K pequeño | K grande |
|---|---|
| Bajo bias, alta varianza | Alto bias, baja varianza |
| Sobreajuste (overfitting) | Subajuste (underfitting) |
| Sensible al ruido | Fronteras más suaves |

### Ventajas y Limitaciones
**Ventajas:** Simple, no hace suposiciones sobre distribución, funciona bien con datos no lineales.  
**Limitaciones:** Lento en predicción O(n·d), sensible a escala, requiere normalización, afectado por la maldición de la dimensionalidad.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import label_binarize
import pickle, os, time

os.makedirs('docs/figs', exist_ok=True)
sns.set_theme(style='whitegrid')
print('Librerías cargadas ✔')

In [ ]:
# ── Cargar datos procesados ───────────────────────────────────────────────
X_train = np.load('data/processed/X_train_sc.npy')
X_test  = np.load('data/processed/X_test_sc.npy')
y_train = np.load('data/processed/y_train.npy')
y_test  = np.load('data/processed/y_test.npy')

class_names = ['Low', 'Medium', 'High']
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Búsqueda del K Óptimo (Elbow Method)

In [ ]:
# ── Evaluar K de 1 a 30 con Cross-Validation ────────────────────────────
k_range = range(1, 31)
cv_scores = []
train_scores = []

print('Evaluando K de 1 a 30...')
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    # Cross-validation en train (5-fold)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    # Score en train
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    if k % 5 == 0:
        print(f'  K={k}: CV Acc={scores.mean():.4f}')

best_k = k_range[np.argmax(cv_scores)]
print(f'\n✔ Mejor K = {best_k} (CV Accuracy = {max(cv_scores):.4f}')

In [ ]:
# ── Gráfica Elbow ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(list(k_range), train_scores, 'o-', label='Train Accuracy', color='steelblue', lw=2)
ax.plot(list(k_range), cv_scores,    's-', label='CV Accuracy (5-fold)', color='tomato', lw=2)
ax.axvline(x=best_k, color='green', linestyle='--', lw=2, label=f'Mejor K={best_k}')
ax.set_xlabel('Valor de K', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Selección del Hiperparámetro K — Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.savefig('docs/figs/knn_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Entrenamiento con el K Óptimo

In [ ]:
knn_final = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='euclidean',
    weights='uniform',   # todos los vecinos igual peso
    algorithm='auto',    # sklearn elige kd-tree o ball-tree automáticamente
    n_jobs=-1
)

start = time.time()
knn_final.fit(X_train, y_train)
train_time = time.time() - start

start = time.time()
y_pred = knn_final.predict(X_test)
pred_time = time.time() - start

print(f'Tiempo de entrenamiento (indexación): {train_time:.2f}s')
print(f'Tiempo de predicción (10,000 muestras): {pred_time:.2f}s')

# Guardar modelo
with open('data/processed/knn_model.pkl', 'wb') as f:
    pickle.dump(knn_final, f)
print('\n✔ Modelo guardado')

## 3. Métricas de Evaluación

In [ ]:
acc   = accuracy_score(y_test, y_pred)
f1_w  = f1_score(y_test, y_pred, average='weighted')
f1_m  = f1_score(y_test, y_pred, average='macro')

print('='*55)
print(f'  KNN (K={best_k}) — Resultados en Test Set')
print('='*55)
print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  F1-Score (weighted): {f1_w:.4f}')
print(f'  F1-Score (macro)   : {f1_m:.4f}')
print('='*55)
print()
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# ── Matriz de Confusión ───────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
axes[0].set_title(f'Matriz de Confusión (KNN K={best_k})', fontweight='bold')
axes[0].set_ylabel('Etiqueta Real')
axes[0].set_xlabel('Etiqueta Predicha')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
axes[1].set_title('Matriz de Confusión (% por clase real)', fontweight='bold')
axes[1].set_ylabel('Etiqueta Real')
axes[1].set_xlabel('Etiqueta Predicha')

plt.tight_layout()
plt.savefig('docs/figs/knn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Curva ROC (One-vs-Rest) ───────────────────────────────────────────────
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_proba    = knn_final.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#4CAF50', '#FF9800', '#F44336']

aucs = []
for i, (cls, col) in enumerate(zip(class_names, colors_roc)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    auc = roc_auc_score(y_test_bin[:, i], y_proba[:, i])
    aucs.append(auc)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'Curva ROC — KNN (K={best_k})\nAUC macro={np.mean(aucs):.3f}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('docs/figs/knn_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Impacto de diferentes métricas de distancia ───────────────────────────
metrics_test = ['euclidean', 'manhattan', 'chebyshev']
results_metrics = {}

for met in metrics_test:
    knn_m = KNeighborsClassifier(n_neighbors=best_k, metric=met, n_jobs=-1)
    knn_m.fit(X_train, y_train)
    acc_m = accuracy_score(y_test, knn_m.predict(X_test))
    results_metrics[met] = acc_m
    print(f'Métrica {met:12s}: Accuracy = {acc_m:.4f}')

print(f'\n✔ Distancia euclidiana es estándar y referenciada en la literatura clásica de KNN')

## 4. Análisis de Resultados

- El modelo KNN logra una accuracy de aproximadamente **XX%** en el conjunto de test.
- La clase **High** suele tener menor recall dado que es la clase minoritaria.
- El AUC por encima de 0.7 en todas las clases indica buena capacidad discriminativa.
- La distancia Euclidiana funciona bien con datos normalizados en este contexto.